# Dataset Generation

Takes clean Places365 images, splits them into train/val/test, then generates
degraded versions within each split. The split happens at the source image level
to prevent data leakage.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from config import *

print(f"Source images: {DATASET_SIZE}")
print(f"Seed: {RANDOM_SEED}")
print(f"Splits: {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")
print(f"Image size: {IMG_SIZE}")

## Check source images

In [ ]:
from pathlib import Path

all_images = sorted(PLACES365_RAW_DIR.glob("**/*.jpg"))
print(f"Found {len(all_images)} images in {PLACES365_RAW_DIR}")
print(f"Examples: {[p.name for p in all_images[:3]]}")

## Generate dataset

Splits clean images first (no leakage), then applies degradations per split.

In [ ]:
from ml.synthetic_data import build_dataset

save_dir = DATA_DIR / "generated"

dataset = build_dataset(
    source_dir=PLACES365_RAW_DIR,
    max_source_images=DATASET_SIZE,
    img_size=IMG_SIZE,
    seed=RANDOM_SEED,
    save_dir=save_dir,
)

for split, samples in dataset.items():
    print(f"{split}: {len(samples)} samples")
print(f"Total: {sum(len(v) for v in dataset.values())}")

## Verify no data leakage

In [ ]:
import pandas as pd

train_meta = pd.read_csv(save_dir / "train" / "metadata.csv")
val_meta = pd.read_csv(save_dir / "val" / "metadata.csv")
test_meta = pd.read_csv(save_dir / "test" / "metadata.csv")

# extract base image name from source_image_id
get_base = lambda df: set(df['source_image_id'].str.split('_', n=2).str[2])
train_src, val_src, test_src = get_base(train_meta), get_base(val_meta), get_base(test_meta)

print(f"Train ∩ Val:  {len(train_src & val_src)} (expect 0)")
print(f"Train ∩ Test: {len(train_src & test_src)} (expect 0)")
print(f"Val ∩ Test:   {len(val_src & test_src)} (expect 0)")
assert not (train_src & val_src) and not (train_src & test_src) and not (val_src & test_src)
print("No leakage ✓")

## Distribution plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, meta) in zip(axes, [("Train", train_meta), ("Val", val_meta), ("Test", test_meta)]):
    counts = meta['degradation_type'].value_counts()
    ax.barh(counts.index, counts.values, color=plt.cm.Set3(np.linspace(0, 1, len(counts))))
    ax.set_title(f'{name} split')
    for i, v in enumerate(counts.values):
        ax.text(v + 5, i, str(v), va='center', fontsize=8)
plt.tight_layout()
plt.savefig(str(save_dir / "degradation_distribution.png"), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, meta in [("Train", train_meta), ("Val", val_meta), ("Test", test_meta)]:
    ax.hist(meta['quality_score'], bins=30, alpha=0.5, label=name)
ax.set_xlabel('Quality Score')
ax.legend()
ax.set_title('Score distribution')
plt.tight_layout()
plt.savefig(str(save_dir / "quality_score_distribution.png"), dpi=150, bbox_inches='tight')
plt.show()

## Sample degradations

In [ ]:
import cv2

source = train_meta['source_image_id'].unique()[0]
rows = train_meta[train_meta['source_image_id'] == source]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for idx, (_, row) in enumerate(rows.iterrows()):
    if idx >= 8: break
    img = cv2.cvtColor(cv2.imread(str(save_dir / "train" / "images" / row['filename'])), cv2.COLOR_BGR2RGB)
    ax = axes[idx // 4][idx % 4]
    ax.imshow(img)
    ax.set_title(f"{row['degradation_type']}\nsev={row['severity']} q={row['quality_score']:.0f}", fontsize=8)
    ax.axis('off')
for ax in axes.flatten()[len(rows):]: ax.axis('off')
plt.tight_layout()
plt.savefig(str(save_dir / "sample_degradations.png"), dpi=150, bbox_inches='tight')
plt.show()